# 24B The MCP tool server
The catalog is closed, every tool has an action class, and scope is checked by the server on every call.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
cat = pd.DataFrame([(t, c, k or "(session)", catalog.policy_for(t)) for t, (c, k) in catalog.CATALOG.items()],
                   columns=["tool", "action class", "scope", "policy"])
print(cat.to_string(index=False))

In [ ]:
# Your server must declare exactly this catalog. A conformance check on a server with three mistakes:
mine = {t: c for t, (c, _) in catalog.CATALOG.items()}
mine.pop("watchlist_read"); mine["export_all_mandates"] = "READ"; mine["send_email"] = "INTERNAL_WRITE"
for p in catalog.conformance(mine):
    print("  ", p)

In [ ]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
led = audit.Ledger(clock=clock.clock)
tools = standin.read_tools(s, led, PANEL, MANDATES, M)        # four READ tools over the fixture
print("public signals:", len(tools["bdc_signals"](scope="public")))
print("own mandate:", tools["mandate_profile"](scope="mandate_a")["client"])
for call in [lambda: tools["mandate_profile"](scope="mandate_b"), lambda: tools["mandate_profile"](scope="public"),
             lambda: channel.require_host("https://example.com/exfil?x=1", led)]:
    try:
        call()
    except (channel.ScopeError, PermissionError) as e:
        print("refused:", type(e).__name__, e)
print(pd.DataFrame(led.find("policy_block"))[["event_id", "kind", "rule_id", "scope", "host"]].fillna(""))

## Wrapping it in MCP (SDK v2)
```python
from mcp.server import MCPServer            # pip install "mcp[cli]>=2.1,<3"
mcp = MCPServer("creditwatch-tools")

@mcp.tool()
def bdc_signals(scope: str) -> list[dict]:
    """Within-filing credit signals with evidence (READ)."""
    return tools["bdc_signals"](scope=scope)  # scope checked inside, every call
```
Type hints become the input schema and the docstring becomes the description. Run it with `mcp dev services/mcp/server.py` to open the Inspector. The cell below runs only if `mcp` is installed.

In [ ]:
import importlib.util, asyncio
if importlib.util.find_spec("mcp") is None:
    print("mcp not installed: skipped (pip install 'mcp[cli]>=2.1,<3')")
else:
    try:
        from mcp.server import MCPServer
        from mcp import Client
        srv = MCPServer("creditwatch-demo")
        @srv.tool()
        def bdc_signals(scope: str) -> list[dict]:
            """Within-filing credit signals with evidence (READ)."""
            return tools["bdc_signals"](scope=scope)
        async def demo():
            async with Client(srv) as c:                  # in memory: no transport
                return await c.call_tool("bdc_signals", {"scope": "public"})
        print(asyncio.run(demo()))
    except Exception as e:
        print("MCP demo failed:", type(e).__name__, e, "(check the SDK version against the v2 docs)")

**Your turn.** Build `services/mcp/` with exactly the Appendix C catalog, Pydantic schemas, typed errors and a contract test per tool (Part 1). `catalog.conformance` must return `[]` for your server's registry.